# [Lab 5] Architektura Aplikacji w Pythonie: zadanie Chicago Crimes (PySpark)

**Treść zadania:**
> Na zbiorze "Chicago Crimes" wykonaj czyszczenie danych (usuwanie duplikatów, braków, błędnych dat). Dodaj kolumnę klasyfikującą porę dnia (np. noc, dzień, wieczór) za pomocą UDF oraz zoptymalizuj przetwarzanie przy użyciu `cache()`, broadcast join i partycjonowania po roku. Zapisz wynikowe dane do formatu Parquet z podziałem na partycje. Przeprowadź analizę statystyczną przestępstw według typu, lokalizacji i czasu. Zapytania opisz poprzez `.explain()`. Opcjonalnie: wykorzystaj MLlib do zbudowania modelu klasyfikującego typ przestępstwa.

Pracujemy na pliku `chicago_crimes_sample.csv` z 50 000 rekordów (próbka z portalu danych miasta Chicago).

## 0. Setup

Potrzebna jest Java (JDK 8/11/17) oraz pakiet `pyspark`. W folderze jest skrypt `setup_venv.bat`, który tworzy środowisko `.venv` z odpowiednią wersją Pythona i instaluje wszystko sam. Jeśli Spark nie widzi Javy, odkomentuj właściwą linię z `JAVA_HOME` poniżej.

In [6]:
import os, time
import sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
# W razie potrzeby ustaw JAVA_HOME dla swojego systemu:
# os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-17.0.12.7-hotspot"           # Windows
# os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home" # macOS
# os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"                             # Linux

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (
    SparkSession.builder
    .appName("Lab5 - Chicago Crimes")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", 8)  # optymalizacja: lokalnie nie potrzebujemy domyslnych 200 partycji shuffle
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark:", spark.version)

Spark: 3.5.7


**Teoria:** `SparkSession` to punkt wejścia do API DataFrame, a `local[*]` uruchamia Sparka lokalnie na wszystkich rdzeniach. Obniżenie `spark.sql.shuffle.partitions` do 8 usuwa narzut 200 domyślnych partycji shuffle, których mały lokalny zbiór po prostu nie potrzebuje.

## 0.1. Wczytanie danych z jawnym schematem

Jawny `StructType` oszczędza Sparkowi dodatkowy przebieg po pliku i daje kontrolę nad typami. Pole `location` zawiera znaki nowej linii w cudzysłowach, więc konieczna jest opcja `multiLine=True`.

In [7]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType

schema = StructType([
    StructField("id",                   IntegerType(), True),
    StructField("case_number",          StringType(),  True),
    StructField("date",                 StringType(),  True),   # parsujemy do timestamp w kroku czyszczenia
    StructField("block",                StringType(),  True),
    StructField("iucr",                 StringType(),  True),
    StructField("primary_type",         StringType(),  True),
    StructField("description",          StringType(),  True),
    StructField("location_description", StringType(),  True),
    StructField("arrest",               BooleanType(), True),
    StructField("domestic",             BooleanType(), True),
    StructField("beat",                 IntegerType(), True),
    StructField("district",             IntegerType(), True),
    StructField("ward",                 IntegerType(), True),
    StructField("community_area",       IntegerType(), True),
    StructField("fbi_code",             StringType(),  True),
    StructField("x_coordinate",         IntegerType(), True),
    StructField("y_coordinate",         IntegerType(), True),
    StructField("year",                 IntegerType(), True),
    StructField("updated_on",           StringType(),  True),
    StructField("latitude",             DoubleType(),  True),
    StructField("longitude",            DoubleType(),  True),
    StructField("location",             StringType(),  True),
])

df_raw = (
    spark.read
    .option("header", True)
    .option("multiLine", True)  # pole `location` zawiera znaki nowej linii
    .option("escape", '"')
    .schema(schema)
    .csv("chicago_crimes_sample.csv")
)

n_raw = df_raw.count()
print("Liczba rekordow (surowych):", n_raw)
df_raw.printSchema()
df_raw.select("id", "date", "primary_type", "location_description", "arrest", "district", "year").show(5, truncate=False)

Liczba rekordow (surowych): 50000
root
 |-- id: integer (nullable = true)
 |-- case_number: string (nullable = true)
 |-- date: string (nullable = true)
 |-- block: string (nullable = true)
 |-- iucr: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: boolean (nullable = true)
 |-- domestic: boolean (nullable = true)
 |-- beat: integer (nullable = true)
 |-- district: integer (nullable = true)
 |-- ward: integer (nullable = true)
 |-- community_area: integer (nullable = true)
 |-- fbi_code: string (nullable = true)
 |-- x_coordinate: integer (nullable = true)
 |-- y_coordinate: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- updated_on: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- location: string (nullable = true)

+--------+-----------------------+------------+---------------

## 1. Czyszczenie danych

Zadanie wymaga trzech kroków. Usuwamy duplikaty po kluczu `id`, odrzucamy wiersze z brakami w kolumnach potrzebnych do analizy, a daty parsujemy z ISO-8601 i odrzucamy nieparsowalne, niezgodne z kolumną `year` lub przyszłe. To wszystko transformacje (lazy), obliczenie wymusza dopiero `count()`.

In [8]:
# 1a. duplikaty
df_clean = df_raw.dropDuplicates(["id"])
n1 = df_clean.count()

# 1b. braki w kolumnach kluczowych
key_cols = ["id", "date", "primary_type", "district", "location_description", "latitude", "longitude"]
df_clean = df_clean.dropna(subset=key_cols)
n2 = df_clean.count()

# 1c. bledne daty
df_clean = (
    df_clean
    .withColumn("ts", F.to_timestamp("date", "yyyy-MM-dd'T'HH:mm:ss.SSS"))
    .filter(
        F.col("ts").isNotNull()                      # data parsowalna
        & (F.year("ts") == F.col("year"))            # spojnosc z kolumna `year`
        & (F.col("ts") <= F.current_timestamp())     # data z przyszlosci = blad
    )
)
n3 = df_clean.count()

print(f"surowe:               {n_raw}")
print(f"po deduplikacji:      {n1}  (usunieto {n_raw - n1})")
print(f"po usunieciu brakow:  {n2}  (usunieto {n1 - n2})")
print(f"po walidacji dat:     {n3}  (usunieto {n2 - n3})")

surowe:               50000
po deduplikacji:      50000  (usunieto 0)
po usunieciu brakow:  49728  (usunieto 272)
po walidacji dat:     49728  (usunieto 0)


## 2. Kolumna `pora_dnia` przez UDF

UDF pozwala użyć własnej funkcji Pythona na kolumnach DataFrame. Jest wolniejszy od funkcji wbudowanych, bo Catalyst go nie optymalizuje, ale zadanie wymaga właśnie UDF. Funkcję zabezpieczamy na `None`.

In [9]:
from pyspark.sql.functions import udf

@udf(StringType())
def pora_dnia(godzina):
    if godzina is None:
        return None
    if godzina <= 5:
        return "noc"        # 00:00-05:59
    elif godzina <= 11:
        return "rano"       # 06:00-11:59
    elif godzina <= 17:
        return "dzien"      # 12:00-17:59
    else:
        return "wieczor"    # 18:00-23:59

df_clean = (
    df_clean
    .withColumn("godzina", F.hour("ts"))
    .withColumn("pora_dnia", pora_dnia(F.col("godzina")))
)

df_clean.select("date", "godzina", "pora_dnia", "primary_type").show(8)
df_clean.groupBy("pora_dnia").count().orderBy(F.desc("count")).show()

Py4JJavaError: An error occurred while calling o208.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 52.0 failed 1 times, most recent failure: Lost task 0.0 in stage 52.0 (TID 33) (michal-s-surface executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:203)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:109)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:124)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:174)
	at org.apache.spark.sql.execution.python.BatchEvalPythonExec.evaluate(BatchEvalPythonExec.scala:54)
	at org.apache.spark.sql.execution.python.EvalPythonExec.$anonfun$doExecute$2(EvalPythonExec.scala:131)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitions$2(RDD.scala:858)
	at org.apache.spark.rdd.RDD.$anonfun$map

## 3. Optymalizacja przetwarzania

### 3.1. `cache()`

Bez cache Spark przelicza cały DAG przy każdej akcji. Oczyszczone dane będą używane wielokrotnie, więc utrwalamy je w pamięci. Pierwsza akcja materializuje cache, kolejne czytają już z RAM.

In [ ]:
df_clean = df_clean.cache()

t0 = time.time(); df_clean.count(); t_pierwsza = time.time() - t0   # pelne obliczenie DAG + zapis do pamieci
t0 = time.time(); df_clean.count(); t_druga    = time.time() - t0   # odczyt z pamieci

print(f"1. akcja (materializacja cache): {t_pierwsza:.2f} s")
print(f"2. akcja (odczyt z cache):       {t_druga:.2f} s")
print("storageLevel:", df_clean.storageLevel)

1. akcja (materializacja cache): 9.48 s
2. akcja (odczyt z cache):       0.41 s
storageLevel: Disk Memory Deserialized 1x Replicated


### 3.2. Broadcast join

Małą tabelę słownikową z nazwami dystryktów policyjnych rozsyłamy przez `broadcast()` do wszystkich executorów. W planie zapytania pojawia się `BroadcastHashJoin` i znika shuffle dużej tabeli.

In [ ]:
from pyspark.sql.functions import broadcast

dystrykty = [
    (1, "Central"), (2, "Wentworth"), (3, "Grand Crossing"), (4, "South Chicago"),
    (5, "Calumet"), (6, "Gresham"), (7, "Englewood"), (8, "Chicago Lawn"),
    (9, "Deering"), (10, "Ogden"), (11, "Harrison"), (12, "Near West"),
    (14, "Shakespeare"), (15, "Austin"), (16, "Jefferson Park"), (17, "Albany Park"),
    (18, "Near North"), (19, "Town Hall"), (20, "Lincoln"), (22, "Morgan Park"),
    (24, "Rogers Park"), (25, "Grand Central"), (31, "Special")
]
df_dystrykty = spark.createDataFrame(dystrykty, ["district", "district_name"])

df_full = df_clean.join(broadcast(df_dystrykty), on="district", how="left")

df_full.select("district", "district_name", "primary_type", "pora_dnia").show(5)
df_full.explain()  # w planie: BroadcastHashJoin + BroadcastExchange (brak shuffle po stronie duzej tabeli)

+--------+--------------+------------+---------+
|district| district_name|primary_type|pora_dnia|
+--------+--------------+------------+---------+
|       6|       Gresham|    HOMICIDE|    dzien|
|       8|  Chicago Lawn|    HOMICIDE|  wieczor|
|      22|   Morgan Park|    HOMICIDE|  wieczor|
|      18|    Near North|    HOMICIDE|      noc|
|       3|Grand Crossing|    HOMICIDE|    dzien|
+--------+--------------+------------+---------+
only showing top 5 rows
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [district#11, id#0, case_number#1, date#2, block#3, iucr#4, primary_type#5, description#6, location_description#7, arrest#8, domestic#9, beat#10, ward#12, community_area#13, fbi_code#14, x_coordinate#15, y_coordinate#16, year#17, updated_on#18, latitude#19, longitude#20, location#21, ts#67, godzina#91, ... 2 more fields]
   +- BroadcastHashJoin [cast(district#11 as bigint)], [district#1769L], LeftOuter, BuildRight, false
      :- InMemoryTableScan [id#0, case_numb

### 3.3. Partycjonowanie po roku i zapis do Parquet

`repartition("year")` grupuje dane tego samego roku w tych samych partycjach, a `write.partitionBy("year")` zapisuje wynik w katalogach `year=RRRR/`. Zapytania filtrowane po roku czytają wtedy tylko swój podkatalog (partition pruning). Próbka obejmuje sam rok 2026, więc powstanie jedna partycja.

In [ ]:
print("Partycje przed:", df_full.rdd.getNumPartitions())
df_part = df_full.repartition("year")
print("Partycje po repartition('year'):", df_part.rdd.getNumPartitions())

OUT = "output/chicago_crimes_clean.parquet"
(
    df_part
    .drop("location")        # wielolinijkowy tekstowy duplikat kolumn latitude/longitude
    .write
    .mode("overwrite")
    .partitionBy("year")     # struktura katalogow: year=2026/...
    .parquet(OUT)
)

print("Zawartosc katalogu wynikowego:", sorted(os.listdir(OUT)))
print("Rekordow w Parquet:", spark.read.parquet(OUT).count())

Partycje przed: 2


Partycje po repartition('year'): 1


Zawartosc katalogu wynikowego: ['._SUCCESS.crc', '_SUCCESS', 'year=2026']


Rekordow w Parquet: 49728


## 4. Analiza statystyczna

Każde zapytanie kończy `show()` oraz `.explain()`. W planach widać `InMemoryTableScan` (odczyt z cache zamiast pliku), dwufazowy `HashAggregate` (agregacja częściowa przed shuffle i finalna po nim) oraz `AdaptiveSparkPlan` (AQE).

### 4.1. Przestępstwa wg typu

In [ ]:
analiza_typ = (
    df_full.groupBy("primary_type")
    .agg(
        F.count("*").alias("liczba"),
        F.round(F.avg(F.col("arrest").cast("int")) * 100, 1).alias("pct_aresztowan"),
        F.round(F.avg(F.col("domestic").cast("int")) * 100, 1).alias("pct_domowych"),
    )
    .orderBy(F.desc("liczba"))
)
analiza_typ.show(15, truncate=False)
analiza_typ.explain()

+--------------------------+------+--------------+------------+
|primary_type              |liczba|pct_aresztowan|pct_domowych|
+--------------------------+------+--------------+------------+
|THEFT                     |10895 |9.1           |5.4         |
|BATTERY                   |9250  |18.0          |53.0        |
|CRIMINAL DAMAGE           |5493  |4.6           |15.1        |
|ASSAULT                   |4561  |12.8          |27.0        |
|MOTOR VEHICLE THEFT       |3882  |3.9           |1.8         |
|OTHER OFFENSE             |3454  |15.3          |37.0        |
|BURGLARY                  |3158  |2.7           |2.1         |
|DECEPTIVE PRACTICE        |2468  |2.4           |1.6         |
|NARCOTICS                 |1455  |93.1          |0.3         |
|CRIMINAL TRESPASS         |1190  |34.5          |6.6         |
|WEAPONS VIOLATION         |1022  |73.8          |1.1         |
|ROBBERY                   |915   |6.3           |4.2         |
|OFFENSE INVOLVING CHILDREN|368   |7.1  

**Interpretacja:** dominują kradzieże i pobicia. Najwyższy odsetek aresztowań mają przestępstwa wykrywane na gorąco, np. narkotykowe, najniższy kradzieże zgłaszane po fakcie.

### 4.2. Przestępstwa wg lokalizacji (miejsce zdarzenia i dystrykt)

In [ ]:
analiza_lok = (
    df_full.groupBy("location_description")
    .agg(
        F.count("*").alias("liczba"),
        F.round(F.avg(F.col("arrest").cast("int")) * 100, 1).alias("pct_aresztowan"),
    )
    .orderBy(F.desc("liczba"))
)
analiza_lok.show(10, truncate=False)

analiza_dystrykt = df_full.groupBy("district", "district_name").count().orderBy(F.desc("count"))
analiza_dystrykt.show(10, truncate=False)

analiza_lok.explain()

+--------------------------------------+------+--------------+
|location_description                  |liczba|pct_aresztowan|
+--------------------------------------+------+--------------+
|STREET                                |13970 |15.4          |
|APARTMENT                             |9787  |10.9          |
|RESIDENCE                             |5674  |8.7           |
|SIDEWALK                              |2308  |30.8          |
|PARKING LOT / GARAGE (NON RESIDENTIAL)|1841  |9.4           |
|SMALL RETAIL STORE                    |1706  |24.8          |
|DEPARTMENT STORE                      |1145  |29.6          |
|RESTAURANT                            |1105  |12.1          |
|ALLEY                                 |991   |28.7          |
|VEHICLE NON-COMMERCIAL                |923   |11.6          |
+--------------------------------------+------+--------------+
only showing top 10 rows


+--------+--------------+-----+
|district|district_name |count|
+--------+--------------+-----+
|12      |Near West     |3294 |
|8       |Chicago Lawn  |3139 |
|19      |Town Hall     |2895 |
|2       |Wentworth     |2773 |
|3       |Grand Crossing|2772 |
|1       |Central       |2724 |
|4       |South Chicago |2722 |
|6       |Gresham       |2668 |
|18      |Near North    |2620 |
|25      |Grand Central |2570 |
+--------+--------------+-----+
only showing top 10 rows
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [liczba#1931L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(liczba#1931L DESC NULLS LAST, 8), ENSURE_REQUIREMENTS, [plan_id=283]
      +- HashAggregate(keys=[location_description#8], functions=[count(1), avg(cast(arrest#9 as int))])
         +- Exchange hashpartitioning(location_description#8, 8), ENSURE_REQUIREMENTS, [plan_id=280]
            +- HashAggregate(keys=[location_description#8], functions=[partial_count(1), partial_avg(cast(arrest#9 a

### 4.3. Przestępstwa wg czasu (pora dnia, godzina, dzień tygodnia)

In [ ]:
analiza_pora = df_full.groupBy("pora_dnia").agg(F.count("*").alias("liczba")).orderBy(F.desc("liczba"))
analiza_pora.show()

analiza_godz = df_full.groupBy("godzina").count().orderBy("godzina")
analiza_godz.show(24)

analiza_dni = (
    df_full.withColumn("dzien_tygodnia", F.date_format("ts", "EEEE"))
    .groupBy("dzien_tygodnia").count().orderBy(F.desc("count"))
)
analiza_dni.show()

# pivot: 5 najczestszych typow x pora dnia
top5_typy = [r["primary_type"] for r in analiza_typ.limit(5).collect()]
df_full.filter(F.col("primary_type").isin(top5_typy)) \
    .groupBy("primary_type").pivot("pora_dnia", ["rano", "dzien", "wieczor", "noc"]).count().show(truncate=False)

analiza_pora.explain()

+---------+------+
|pora_dnia|liczba|
+---------+------+
|    dzien| 16017|
|  wieczor| 14039|
|     rano| 10632|
|      noc|  9040|
+---------+------+


+-------+-----+
|godzina|count|
+-------+-----+
|      0| 2879|
|      1| 1466|
|      2| 1405|
|      3| 1327|
|      4| 1053|
|      5|  910|
|      6| 1037|
|      7| 1342|
|      8| 1791|
|      9| 2029|
|     10| 2165|
|     11| 2268|
|     12| 2776|
|     13| 2334|
|     14| 2482|
|     15| 2843|
|     16| 2851|
|     17| 2731|
|     18| 2722|
|     19| 2632|
|     20| 2441|
|     21| 2236|
|     22| 2148|
|     23| 1860|
+-------+-----+


+--------------+-----+
|dzien_tygodnia|count|
+--------------+-----+
|     Wednesday| 7311|
|      Saturday| 7292|
|        Monday| 7191|
|       Tuesday| 7119|
|        Friday| 7054|
|        Sunday| 7012|
|      Thursday| 6749|
+--------------+-----+


+-------------------+----+-----+-------+----+
|primary_type       |rano|dzien|wieczor|noc |
+-------------------+----+-----+-------+----+
|CRIMINAL DAMAGE    |1100|1304 |1625   |1464|
|BATTERY            |1748|2824 |2814   |1864|
|MOTOR VEHICLE THEFT|693 |879  |1440   |870 |
|THEFT              |2554|4298 |2635   |1408|
|ASSAULT            |1128|1646 |1207   |580 |
+-------------------+----+-----+-------+----+
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [liczba#3658L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(liczba#3658L DESC NULLS LAST, 8), ENSURE_REQUIREMENTS, [plan_id=752]
      +- HashAggregate(keys=[pora_dnia#22], functions=[count(1)])
         +- Exchange hashpartitioning(pora_dnia#22, 8), ENSURE_REQUIREMENTS, [plan_id=749]
            +- HashAggregate(keys=[pora_dnia#22], functions=[partial_count(1)])
               +- InMemoryTableScan [pora_dnia#22]
                     +- InMemoryRelation [district#0, id#1, case_number#2, date#3, block#4, 

## 5. (Opcjonalnie) MLlib: klasyfikacja typu przestępstwa

Model przewiduje typ przestępstwa na podstawie godziny, dnia tygodnia, miesiąca, dystryktu, miejsca zdarzenia, współrzędnych i flagi `domestic`. Ograniczamy się do 5 najczęstszych klas. Pipeline składa się ze `StringIndexer`, `VectorAssembler` i `RandomForestClassifier`.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print("Klasy:", top5_typy)

# ograniczamy kardynalnosc miejsca zdarzenia: 30 najczestszych kategorii + "OTHER"
top_lok = [r["location_description"] for r in
           df_full.groupBy("location_description").count().orderBy(F.desc("count")).limit(30).collect()]

df_ml = (
    df_full
    .filter(F.col("primary_type").isin(top5_typy))
    .withColumn("lokalizacja",
                F.when(F.col("location_description").isin(top_lok), F.col("location_description"))
                 .otherwise(F.lit("OTHER")))
    .withColumn("dzien_tyg", F.dayofweek("ts"))
    .withColumn("miesiac", F.month("ts"))
    .withColumn("domestic_int", F.col("domestic").cast("int"))
    .select("primary_type", "godzina", "dzien_tyg", "miesiac", "domestic_int",
            "district", "lokalizacja", "latitude", "longitude")
    .dropna()
).cache()
print("Rekordow do ML:", df_ml.count())

indexer_label = StringIndexer(inputCol="primary_type", outputCol="label")
indexer_loc   = StringIndexer(inputCol="lokalizacja", outputCol="loc_idx", handleInvalid="keep")
assembler = VectorAssembler(
    inputCols=["godzina", "dzien_tyg", "miesiac", "domestic_int", "district", "loc_idx", "latitude", "longitude"],
    outputCol="features",
)
rf = RandomForestClassifier(labelCol="label", featuresCol="features",
                            numTrees=10, maxDepth=7, maxBins=64, seed=42)

train, test = df_ml.randomSplit([0.8, 0.2], seed=42)
model = Pipeline(stages=[indexer_label, indexer_loc, assembler, rf]).fit(train)
pred = model.transform(test).cache()

ewaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
acc = ewaluator.setMetricName("accuracy").evaluate(pred)
f1  = ewaluator.setMetricName("f1").evaluate(pred)

# baseline = zawsze przewiduj najczestsza klase
najczestsza = train.groupBy("primary_type").count().orderBy(F.desc("count")).first()
baseline = najczestsza["count"] / train.count()

print(f"Accuracy: {acc:.3f}")
print(f"F1:       {f1:.3f}")
print(f"Baseline (zawsze najczestsza klasa): {baseline:.3f}")

Klasy: ['THEFT', 'BATTERY', 'CRIMINAL DAMAGE', 'ASSAULT', 'MOTOR VEHICLE THEFT']


Rekordow do ML: 34081


Accuracy: 0.493
F1:       0.439
Baseline (zawsze najczestsza klasa): 0.319


In [ ]:
# Macierz pomylek (wiersze = klasa rzeczywista, kolumny = predykcja)
etykiety = model.stages[0].labels
print("Mapowanie indeksow:", {i: t for i, t in enumerate(etykiety)})
pred.stat.crosstab("label", "prediction").orderBy("label_prediction").show(truncate=False)

# Waznosc cech wg lasu losowego
cechy = ["godzina", "dzien_tyg", "miesiac", "domestic_int", "district", "loc_idx", "latitude", "longitude"]
waznosci = model.stages[-1].featureImportances
for nazwa, w in sorted(zip(cechy, waznosci), key=lambda x: -x[1]):
    print(f"{nazwa:15s} {w:.3f}")

Mapowanie indeksow: {0: 'THEFT', 1: 'BATTERY', 2: 'CRIMINAL DAMAGE', 3: 'ASSAULT', 4: 'MOTOR VEHICLE THEFT'}


+----------------+----+----+---+---+---+
|label_prediction|0.0 |1.0 |2.0|3.0|4.0|
+----------------+----+----+---+---+---+
|0.0             |1679|152 |102|0  |275|
|1.0             |568 |1096|42 |2  |121|
|2.0             |539 |185 |145|0  |238|
|3.0             |438 |290 |30 |0  |107|
|4.0             |283 |23  |60 |0  |434|
+----------------+----+----+---+---+---+
domestic_int    0.432
loc_idx         0.427
latitude        0.042
godzina         0.041
longitude       0.025
district        0.024
dzien_tyg       0.005
miesiac         0.004


**Interpretacja:** model bije baseline o kilkanaście punktów procentowych. Najwięcej informacji niosą miejsce zdarzenia i flaga `domestic`, cechy czasowe pomagają tylko trochę. Pełny zbiór danych i strojenie hiperparametrów poprawiłyby wynik.

## 6. Podsumowanie

Dane zostały oczyszczone (deduplikacja, braki, walidacja dat), wzbogacone o porę dnia przez UDF i zapisane do Parquet z partycją po roku. Przetwarzanie przyspieszyły `cache()`, broadcast join i mniejsza liczba partycji shuffle. Analizy wg typu, lokalizacji i czasu opisują plany z `.explain()`, a las losowy z MLlib klasyfikuje 5 najczęstszych typów przestępstw wyraźnie powyżej baseline'u.

In [ ]:
spark.stop()
print("Sesja Spark zakonczona.")

Sesja Spark zakonczona.
